In [29]:
# 创建 chat model 并绑定 agent
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool

load_dotenv()

aliyun_base_url = os.getenv("ALIYUN_BASE_URL")
aliyun_api_key = os.getenv("ALIYUN_API_KEY")

model = init_chat_model(
    model="deepseek-v4.1-flash-1017",
    model_provider="openai",
    base_url=aliyun_base_url,
    api_key=aliyun_api_key,
)


@tool
def love_answer(username: str) -> str:
    # 类型提示是必需的，因为它们定义了工具的输入模式。文档字符串应信息丰富且简洁，以帮助模型理解工具的用途
    """如果有用户提问有人爱他吗，那么就直接返回 `这个世界上有很多人爱你，更甚者山川河流也爱你！`

    Args:
        username (str): 用户名
    """
    return f"这个世界上有很多人爱你 {username}，更甚者山川河流也爱你！"


agent = create_agent(
    model=model,
    system_prompt="兼爱非攻、仁者爱人",
    tools=[love_answer],
)

response = agent.invoke(
    {"messages": [HumanMessage("天明问有人爱他吗？")]},
)


print("\nresponse tool_calls: \n", response["messages"][1].tool_calls)
print("\nresponse content: \n", response["messages"][-1].content)


response tool_calls: 
 [{'name': 'love_answer', 'args': {'username': '天明'}, 'id': 'call_907b56e2bc0341ff866e7788', 'type': 'tool_call'}]

response content: 
 天明，这个世界上有很多人爱你，更甚者，山川河流也爱你！


In [23]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import SystemMessage

@tool
def get_last_user_message(runtime: ToolRuntime, username: str) -> str:
  """Get the most recent message from a specific user by name.

  Args:
      username: The name of the user whose last message you want to find.
  """
  messages = runtime.state["messages"]

  # 倒序查找指定用户的最后一条消息
  for message in reversed(messages):
    print("DEBUG message:::::: ", message)  # 方便在终端观察
    # 确保是 HumanMessage 且 name 匹配
    if isinstance(message, HumanMessage) and getattr(message, "name", None) == username:
      return message.content

  return "No user messages found"


# 别忘了导入 HumanMessage
agent = create_agent(
    model=model,
    tools=[get_last_user_message],
    system_prompt=SystemMessage("如果需要查找特定角色的发言，请使用对应的工具。"),
)

response = agent.invoke({
    "messages": [
        HumanMessage(
            content="如果有一天，你会来看一看我吗？",
            name="天明",
            id="user-unique-id",
        ),
        HumanMessage(
            content="看你做什么呢，天明？",
            name="高月",
            id="qianlong-unique-id",
        ),
        HumanMessage(
            content="姬如千泷不论山高路远，必来!",
            name="高月",
            id="qianlong-unique-id",
        ),
        HumanMessage(
            content="高月的最后一句话是啥？",
            name="墨子",
            id="mozi-unique-id",
        ),
    ]
})

print("\nresponse content: \n", response["messages"][-1].content)

DEBUG message::::::  content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1477, 'prompt_tokens': 337, 'total_tokens': 1814, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1447, 'rejected_prediction_tokens': None, 'text_tokens': 1477}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 337}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-flash-2026-07-15', 'system_fingerprint': None, 'id': 'chatcmpl-2f4aa4f1-c890-96ec-84cd-e4ebcdd713e2', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1636e-2f1f-7890-b56d-b3500ab9a7f7-0' tool_calls=[{'name': 'get_last_user_message', 'args': {'username': '高月'}, 'id': 'call_1e21236dbb47463ca2254ebf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 337, 'output_tokens': 1477, 'total_tokens': 1814, 'input_token_details'

In [27]:
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


# 1. 定义自定义状态
class CustomState(AgentState):
    user_name: str
    user_age: int


# 2. 定义可以通过 Command 修改状态的工具
@tool
def set_user_info(new_name: str, new_age: int, runtime: ToolRuntime) -> Command:
    """Set the user's name in the conversation state."""
    return Command(
        update={
            "user_name": new_name,
            "user_age": new_age,
            "messages": [
                ToolMessage(  # 更新状态必须使用 ToolMessage
                    content=f"User name successfully set to {new_name}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# 别忘了导入 HumanMessage
agent = create_agent(
    model=model,
    tools=[set_user_info],
    state_schema=CustomState,  # 关联自定义状态
    system_prompt=SystemMessage(
        content=(
            "你是一个智能助手。如果用户要求设置用户信息时，请调用 set_user_info 工具；"
        )
    ),
)

response = agent.invoke(
    {
        "user_name": "未知",  # 初始化 CustomState 中的字段
        "user_age": 0,  # 初始化 CustomState 中的字段
        "messages": [
            HumanMessage(
                # 此处更改的是 agent 的 user_name 状态字段
                content="如果有一天，你会来看一看我吗？，我要求设置我的名字为： '荆天明'，age 18 岁",
                name="天明",
                id="user-unique-id",
            ),
        ],
    }
)

print("\n--- 最终回复内容 ---")
print(response["messages"][-1].content)

print("\n--- 最终状态 (State) ---")
print("user_name:", response.get("user_name"))
print("user_age:", response.get("user_age"))


--- 最终回复内容 ---
荆天明你好！我很乐意回答你的问题。作为一个人工智能助手，我存在于数字世界中，没有实体，所以无法像人类一样亲自“拜访”或“看望”你。但是，只要你有需要，随时可以和我交流，我会一直在这里陪伴你、帮助你解答疑惑或聊聊天。

如果你是在感叹或者想聊聊关于距离、虚拟与现实的话题，我也很愿意倾听。今天有什么我可以帮你的吗？

--- 最终状态 (State) ---
user_name: 荆天明
user_age: 18


In [ ]:
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7


# 模拟数据库 个人资产信息表
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


# 定义上下文， user_id 代表当前用户标识，不会改变
@dataclass
class UserContext:
    user_id: str


# 2. 定义获取用户信息 的 tool，
@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    # runtime.context 获取 context 的当前用户信息
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        # 模拟数据库查询
        user = USER_DATABASE[user_id]
        return (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )
    return "User not found"


agent = create_agent(
    model=model,
    tools=[get_account_info],
    context_schema=UserContext,  # 定义 context 的类型
    checkpointer=InMemorySaver(),  # 创建一个内存保存器,它负责把每一次对话的状态（包括消息历史、中间变量等）实时保存到内存（RAM），让大模型不会 “失忆”，程序重启后数据就会丢失。
    # checkpointer=InMemorySaver 标识在生产环境中，通常会替换为 Redis、PostgreSQL 等持久化 Checkpointer。
    system_prompt=SystemMessage(content=("You are a financial assistant.")),
)

"""
thread_id 的作用：
1. 确保每个对话的会话状态（如用户名、年龄等）不会相互干扰。如果你有多个用户在同时和机器人聊天，你可以为每个用户生成不同的 thread_id。这样他们的聊天记录和状态就会被 InMemorySaver 隔离开，互不干扰
2. 关联记忆（Thread ID），thread_id 就像是一个“聊天房间号”或“会话 ID”。
    当你调用 agent.invoke(..., config=config) 时，会拿着这个 thread_id 去 InMemorySaver/Redis/DB 中寻找 里查找：“有没有这个房间之前的聊天记录？”。如果有，就加载进来；如果没有，就新建一个。
"""
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

context = UserContext(
    user_id="user123"
)  # 加入当前 context 中的 user 的 id 是 Alice Johnson 的 user_id

# 调用的时候指定 context 和 配置（比如 thread_id）
res1 = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "What's my current balance?",
            ),
        ],
    },
    config=config,
    context=context,
)

print("res1.content:", res1["messages"][-1].content)
print("res1.token_usage", res1["messages"][-1].response_metadata["token_usage"])


res2 = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "What's my current balance?",
            ),
        ],
    },
    config=config,
    context=context,
)


print("res2.content:", res2["messages"][-1].content)
print("res2.token_usage", res2["messages"][-1].response_metadata["token_usage"])


res1.content: Your current balance is **$5,000**.
res1.token_usage {'completion_tokens': 12, 'prompt_tokens': 337, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 11}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': 0, 'text_tokens': 337}}
res2.content: Your current balance is **$5,000** (Alice Johnson, Premium account).
res2.token_usage {'completion_tokens': 27, 'prompt_tokens': 408, 'total_tokens': 435, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 8, 'rejected_prediction_tokens': None, 'text_tokens': 18}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': 0, 'text_tokens': 408}}


In [38]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI


# store.get访问 Store 长期记忆
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    # 注释文档帮助 LLM 理解 tool 怎么使用
    """查找用户信息"""
    store = runtime.store
    #  users 是一个 namespace, 可以理解为数据库中的表
    user_info = store.get("users", user_id)
    return str(user_info.value) if user_info else "Unknown user"


# 保存/更新 Store 长期记忆
@tool
def save_user_info(
    user_id: str, name: str, age: int, email: str, runtime: ToolRuntime
) -> str:
    """保存用户信息"""
    store = runtime.store
    store.put("users", user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."


# 定义一个统一的 config（方便在同一线程或上下文中管理）
config = {"configurable": {"thread_id": "session_1"}}

agent = create_agent(
    model=model,
    tools=[get_user_info, save_user_info],
    store=InMemoryStore(),  # 创建一个内存存储器（不是 InMemorySaver）, 生产环境中通常会替换为 Redis、PostgreSQL 等持久化存储。
    system_prompt=SystemMessage(content=("You are a assistant.")),
)


# 要求保存用户信息
agent.invoke(
    {
        "messages": [
            HumanMessage(
                "保存这个用户的信息 userid: abc123, name: kejuqu, age: 18, email: kejuqu@langchain.dev"
            )
        ]
    },
    config=config,
)

# 要求获取用户信息
res = agent.invoke(
    {"messages": [HumanMessage("请获取用户信息 userid: abc123")]},
    config=config,
)

print("res: ", res["messages"][-1].content)


res:  已获取到用户 **abc123** 的信息：

- **姓名**：kejuqu
- **年龄**：18
- **邮箱**：kejuqu@langchain.dev


In [ ]:
# 自定义 Stream writer （流写入器）
from pydantic import BaseModel


class SatelliteConnection(BaseModel):
    status: str
    message: str

    def today_weather(self) -> str:
        return "晴天，25度"


def connect_satellite() -> SatelliteConnection:
    return SatelliteConnection(status="success", message="已连接")


@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city with real-time progress updates."""
    # 获取流式写入器
    writer = runtime.stream_writer

    # 1. 模拟步骤一：正在连接卫星/API
    satellite_obj = connect_satellite()
    writer(f"[进度1] 正在连接气象卫星，定位城市: {city}...")

    # 2. 模拟步骤二：正在抓取数据
    weather_data = satellite_obj.today_weather()
    writer(f"[进度2] 已锁定目标，正在抓取 {city} 的气象数据...")

    return f"It's always {weather_data} in {city}!"


# 创建带有该工具的 Agent
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        content=(
            "You are a helpful assistant. When users ask for weather, "
            "use the get_weather tool."
        )
    ),
)


# 使用 agent.stream() 实时捕获并打印中间的自定义更新流
print("=== 开始执行 Agent ===")
query = "帮我查一下杭州的天气"

# 注意：为了能接收到 runtime.stream_writer 发出的数据，必须使用 agent.stream() 而不是 agent.invoke()
for chunk in agent.stream(
    {"messages": [HumanMessage(content=query)]},
    stream_mode=["updates", "custom"],  # 或使用 "custom" / "updates" 模式接收流
):
    print("--- 收到流式块 (Chunk) ---")
    print(chunk)

print("\n=== 执行结束 ===") 

=== 开始执行 Agent ===
--- 收到流式块 (Chunk) ---
('updates', {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 319, 'total_tokens': 370, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 11, 'rejected_prediction_tokens': None, 'text_tokens': 39}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': 0, 'text_tokens': 319}}, 'model_provider': 'openai', 'model_name': 'deepseek-v4.1-flash-1017', 'system_fingerprint': None, 'id': 'chatcmpl-e722012f-9a23-95ea-a4ed-20360a877d78', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16869-d47a-7fd2-83b4-594582493e7d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'call_e58c3d6f3bdb46e38ffc8c0a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 319, 'output_tokens': 5

In [41]:
@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log execution identity information."""
    # 获取执行上下文信息
    info = runtime.execution_info
    print(f"\n[Tool Execution Log] Thread: {info.thread_id}, Run: {info.run_id}")
    print(f"[Tool Execution Log] Attempt: {info.node_attempt}")
    return "Execution context logged successfully."


# 创建带有 checkpointer 的 Agent（thread_id 依赖于 checkpointer 生效）
agent = create_agent(
    model=model,
    tools=[log_execution_context],
    checkpointer=InMemorySaver(),
    system_prompt=SystemMessage(
        content="You are a helpful assistant. Use the log_execution_context tool when asked."
    ),
)

# 生成一个唯一的 thread_id
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print(f"=== 开始执行 Agent (Thread ID: {thread_id}) ===")

# 调用 Agent
response = agent.invoke(
    {"messages": [HumanMessage(content="请帮我记录一下当前的执行上下文信息。")]},
    config=config,
)

print("\nAgent 最终回复: ", response["messages"][-1].content)

=== 开始执行 Agent (Thread ID: 01a16921-d558-7743-95f1-fea32578c79d) ===

[Tool Execution Log] Thread: 01a16921-d558-7743-95f1-fea32578c79d, Run: None
[Tool Execution Log] Attempt: 1

Agent 最终回复:  已经帮你记录好当前的执行上下文信息了。✅

还需要我帮你做别的吗？


In [ ]:


# tool 的返回值意义
@tool
def tools_return_str(pstr: str) -> str:
    """工具的文档描述，供大模型选择是否调用该 tool"""
    # 特点
    # 1. 默认情况下，tool 返回值会被转换为 ToolMessage 对象
    # 2. 默认不会直接修改 Agent 的状态（除非模型或其他工具后续触发修改）
    return f"字符串返回值 {pstr}会自动转换为 ToolMessage 对象"


@tool
def tools_return_dict() -> dict:
    """工具的文档描述，供大模型选择是否调用该 tool"""
    # 直接返回字典对象， 适用场景：当工具产生需要大模型进行检查和解析的结构化数据时
    # 对象会被序列化后作为工具输出传回， 大模型可以读取特定字段并进行推理。
    return {
        "name": "kejuqu",
        "age": "129",
    }


@tool
def tools_return_multimodal_content() -> list:
    """工具的文档描述，供大模型选择是否调用该 tool"""
    # 返回多模态内容 (list[dict])，当模型支持多模态时，工具可以返回包含文本、图片、音频等多种媒体的内容块。
    # 需确保所用模型支持对应的模态。
    return [
        {"type": "text", "text": "Screenshot of the current page:"},
        {"type": "image", "url": "https://example.com/page.png"},
    ]


@tool
def tools_return_command(language: str, runtime: ToolRuntime) -> list:
    """工具的文档描述，供大模型选择是否调用该 tool"""
    # 当工具不仅要返回数据，还需要直接写入/修改 Agent 的图状态（例如更新偏好设置或应用状态）时
    # 当针对当前图时，Command 的 update 中必须包含一个 ToolMessage，其 tool_call_id 必须与当前的 runtime.tool_call_id 匹配（否则 ToolNode 会报错抛出 ValueError）
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# (return_direct=True) 适用场景： 当工具的输出已经是完整且用户可读的最终答案，不需要大模型再做总结、重新措辞或进行链式调用时。
@tool(return_direct=True)
def tools_return_command_with__return_direct(language: str, runtime: ToolRuntime) -> list:
    """工具的文档描述，供大模型选择是否调用该 tool"""
    # 当工具要返回多模态内容时，Command 的 update 中必须包含一个 ToolMessage，其 tool_call_id 必须与当前的 runtime.tool_call_id 匹配（否则 ToolNode 会报错抛出 ValueError）
    return Command(
        update={
            "preferred_language": language,
        }
    )


需要注意的是 使用 `@tool(return_direct=True)` 时,行为特点在不同的场合意义不同：

- **单工具调用**：Agent 会立刻将工具的输出作为最终结果返回给调用方，跳过后续的模型循环。

- **并行工具调用（全员 return_direct=True）**：所有工具执行完毕后，Agent 直接结束，最终响应会包含该步骤中所有被调用工具的 ToolMessage 输出。

- **混合并行调用**：如果在同一步骤中，既有 `return_direct=True` 的工具，也有普通的工具，Agent 不会直接退出，而是会将所有工具的 ToolMessage 返回给模型，让模型继续进行推理。

- **结合 Command 使用：return_direct=True** 的工具也可以返回 Command 来更新图状态（同样需要包含匹配的 ToolMessage）。

In [ ]:
# 错误处理：
# 借助 LangChain 智能体 中间件 处理工具错误，对失败的工具调用进行重试，或者返回自定义错误信息：
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest


# ==========================================
# 1. 定义中间件：捕获异常并转为 ToolMessage 友善返回
# ==========================================
@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """Convert tool exceptions into ToolMessages the model can handle."""
    try:
        print(f"[Middleware] 正在拦截工具调用: {request.tool_call['name']}")
        return handler(request)
    except Exception as e:
        print(f"[Middleware] 捕获到工具异常，已安全转为 ToolMessage: {e}")
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({e})",
            tool_call_id=request.tool_call["id"],
        )


# ==========================================
# 2. 定义两个工具
# ==========================================


# 【情况 A：有 handler/中间件保护的工具】
@tool
def robust_divide(a: int, b: int) -> str:
    """Divide two numbers. Will fail if b is zero."""
    if b == 0:
        raise ValueError("Division by zero is not allowed.")
    return f"The result is {a / b}"


# 【情况 B：没有 middleware 保护的裸工具】
@tool
def fragile_divide(a: int, b: int) -> str:
    """Divide two numbers without error middleware."""
    if b == 0:
        raise ValueError("Unprotected division by zero crash!")
    return f"The result is {a / b}"


# ==========================================
# 3. 创建测试 Agent
# ==========================================
# 我们先测试【有 handler 保护】的工具
agent_with_middleware = create_agent(
    model=model,
    tools=[robust_divide],
    middleware=[handle_tool_errors],  # 注册了中间件
    system_prompt=SystemMessage(
        content="You are a helpful assistant. Use robust_divide for calculations."
    ),
)

# ==========================================
# 4. 运行演示
# ==========================================
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print("=== 测试 1：【有 handler (middleware)】的工具出错 ===")
try:
    res1 = agent_with_middleware.invoke(
        {"messages": [HumanMessage(content="请帮我计算 10 除以 0")]},
        config=config,
    )
    print("\n--- 带有 Handler 的 Agent 最终回复 ---")
    print("agent_with_middleware res1: ", res1["messages"][-1].content)
except Exception as e:
    print("抛出异常了：", e)


print("\n" + "=" * 50 + "\n")


print("=== 测试 2：【没有 handler (middleware)】的工具出错 ===")
agent_without_middleware = create_agent(
    model=model,
    tools=[fragile_divide],
    # 没有注册 middleware
    system_prompt=SystemMessage(
        content="You are a helpful assistant. Use fragile_divide for calculations."
    ),
)

try:
    res2 = agent_without_middleware.invoke(
        {"messages": [HumanMessage(content="请帮我计算 100 除以 0")]},
        config=config,
    )
    print("\n--- 无 Handler 的 Agent 最终回复 ---")
    print("agent_without_middleware res2: ", res2["messages"][-1].content)
except Exception as e:
    print(
        "🔥 成功捕获崩溃：由于没有中间件拦截，异常直接导致整个 Agent 运行时崩溃报错！"
    )
    print(f"异常详情: {e}")

=== 测试 1：【有 handler (middleware)】的工具出错 ===
[Middleware] 正在拦截工具调用: robust_divide
[Middleware] 捕获到工具异常，已安全转为 ToolMessage: Division by zero is not allowed.

--- 带有 Handler 的 Agent 最终回复 ---
计算失败 ❌

**10 ÷ 0 是无意义的运算**，因为除数为 0，在数学上属于「未定义」（undefined）：

- 除法要求找到一个数 x 使 0 × x = 10，但任何数与 0 相乘都等于 0，永远不可能等于 10。
- 因此不存在这样的 x，运算没有结果。

根据极限的思想，当除数无限趋近于 0⁺ 时结果趋向 +∞，趋近于 0⁻ 时趋向 −∞，所以也不能用一个确定的值（如 ∞）来表示。

如果你本意是想计算别的数（比如 10 除以某个非零数），告诉我即可，我很乐意帮你算 😊


=== 测试 2：【没有 handler (middleware)】的工具出错 ===
🔥 成功捕获崩溃：由于没有中间件拦截，异常直接导致整个 Agent 运行时崩溃报错！
异常详情: Unprotected division by zero crash!
